# 🎱 Powerball ML — a full machine-learning pipeline on lottery data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jamelski1/pbml/blob/claude/powerball-prediction-ml-fxhi8i/notebooks/powerball_ml.ipynb)

> ## ⚠️ Read this first
> Powerball draws are **independent random events** — gravity-pick machines with no memory.
> Past draws contain **zero information** about future draws, so *no* model (this one included)
> can genuinely predict winning numbers. This notebook is an **educational project**: we use
> Powerball data as a fun vehicle to learn a *real*, end-to-end ML workflow — data ingestion,
> EDA, statistical testing, Transformer training on a GPU, honest backtesting, and publishing
> to Hugging Face. The backtest section proves the model ties with random guessing — that
> result *is the lesson*.

**What you'll build, top to bottom:**
1. 📥 Load & clean real Powerball history (NY State Open Data)
2. 📊 Exploratory analysis — hot/cold numbers, gaps, co-occurrence
3. 🎲 A chi-square test showing the draws are statistically uniform
4. 🧠 A Transformer sequence model trained on GPU (PyTorch + mixed precision)
5. ⚖️ A walk-forward backtest vs. a random baseline (the honest reveal)
6. 🎟️ A number generator you can actually play
7. 🤗 Push the trained model to the Hugging Face Hub

**Runtime setup:** `Runtime → Change runtime type → T4 GPU` (free tier) or A100/L4 if you
have Colab Pro. The dataset is tiny so even CPU works — the GPU is here so you learn the
GPU workflow. (TPU runtimes need PyTorch/XLA, a different setup — stick with GPU for this one.)

## 0 · Setup

Colab ships with PyTorch, pandas, matplotlib and scipy preinstalled. We only add
`huggingface_hub` for the final publishing step, then detect our compute device.

In [ ]:
%pip install -q huggingface_hub

import math, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from scipy import stats

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} — using device: {device}")
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    !nvidia-smi -L
else:
    print("Tip: Runtime → Change runtime type → T4 GPU to see the GPU workflow in action.")

## 1 · Load & clean the data

New York State publishes every Powerball draw as open data — a clean CSV, no API key needed.

**Crucial domain detail:** Powerball has changed its rules several times. Since
**October 7, 2015** the game draws 5 white balls from **1–69** and a Powerball from **1–26**.
Before that, the ranges were different (e.g. white 1–59) — so older draws could *never*
contain a 60-69 white ball. Mixing eras would poison every frequency statistic, so we filter
to the current-rules era. This is a classic real-world lesson: **know your data's regime changes.**

In [ ]:
DATA_URL = "https://data.ny.gov/api/views/d6yy-54nr/rows.csv?accessType=DOWNLOAD"

try:
    raw = pd.read_csv(DATA_URL)
    print(f"Downloaded {len(raw)} draws from NY Open Data")
except Exception as e:
    print("Download failed:", e)
    print("Fallback: upload a CSV with columns 'Draw Date' and 'Winning Numbers'")
    from google.colab import files
    uploaded = files.upload()
    raw = pd.read_csv(next(iter(uploaded)))

raw.head()

In [ ]:
# Parse "Winning Numbers" ("NN NN NN NN NN PP") into structured columns.
df = raw.copy()
df["Draw Date"] = pd.to_datetime(df["Draw Date"])
nums = df["Winning Numbers"].str.split(expand=True).astype(int)
df[["w1", "w2", "w3", "w4", "w5"]] = np.sort(nums.iloc[:, :5].values, axis=1)
df["pb"] = nums.iloc[:, 5]

# Keep only the current-rules era (white 1-69, Powerball 1-26).
RULES_CHANGE = pd.Timestamp("2015-10-07")
df = (df[df["Draw Date"] >= RULES_CHANGE]
      .sort_values("Draw Date").reset_index(drop=True))

WHITE_COLS = ["w1", "w2", "w3", "w4", "w5"]
N_WHITE, N_PB = 69, 26

# Sanity checks — never trust data you didn't verify.
assert df[WHITE_COLS].values.min() >= 1 and df[WHITE_COLS].values.max() <= N_WHITE
assert df["pb"].min() >= 1 and df["pb"].max() <= N_PB

print(f"{len(df)} draws in the current-rules era "
      f"({df['Draw Date'].min().date()} → {df['Draw Date'].max().date()})")
df[["Draw Date"] + WHITE_COLS + ["pb"]].tail()

## 2 · Exploratory data analysis

First rule of any ML project: **look at your data.** We'll chart number frequencies,
find the "hot" and "cold" numbers, and check how long each number typically goes unseen.

Keep a skeptical eye: with ~1,000+ draws of 5 balls from 69, *some* numbers will look hot
purely by chance. Section 3 tests whether these patterns are real.

In [ ]:
white_counts = pd.Series(df[WHITE_COLS].values.ravel()).value_counts().reindex(
    range(1, N_WHITE + 1), fill_value=0)
pb_counts = df["pb"].value_counts().reindex(range(1, N_PB + 1), fill_value=0)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
expected_white = len(df) * 5 / N_WHITE
axes[0].bar(white_counts.index, white_counts.values, color="#4C72B0")
axes[0].axhline(expected_white, color="#C44E52", ls="--",
                label=f"expected if uniform ≈ {expected_white:.0f}")
axes[0].set_title("White ball frequency (1–69)"); axes[0].legend()

expected_pb = len(df) / N_PB
axes[1].bar(pb_counts.index, pb_counts.values, color="#DD8452")
axes[1].axhline(expected_pb, color="#C44E52", ls="--",
                label=f"expected if uniform ≈ {expected_pb:.0f}")
axes[1].set_title("Powerball frequency (1–26)"); axes[1].legend()
plt.tight_layout(); plt.show()

print("🔥 hottest whites:", list(white_counts.nlargest(5).index))
print("🧊 coldest whites:", list(white_counts.nsmallest(5).index))

In [ ]:
# Gap analysis: draws since each white number last appeared.
last_seen = {n: None for n in range(1, N_WHITE + 1)}
for i, row in enumerate(df[WHITE_COLS].values):
    for n in row:
        last_seen[n] = i
gaps = pd.Series({n: len(df) - 1 - i if i is not None else len(df)
                  for n, i in last_seen.items()}).sort_values(ascending=False)

plt.figure(figsize=(14, 4))
plt.bar(gaps.index.astype(str), gaps.values, color="#55A868")
plt.title("Draws since each white ball last appeared (most 'overdue' first)")
plt.xticks(fontsize=7); plt.tight_layout(); plt.show()
print("Most 'overdue':", list(gaps.head(5).index),
      "— the gambler's fallacy says play these. The gambler's fallacy is wrong.")

In [ ]:
# Pair co-occurrence: how often does each pair of white balls land together?
co = np.zeros((N_WHITE, N_WHITE), dtype=int)
for row in df[WHITE_COLS].values:
    for a in row:
        for b in row:
            if a != b:
                co[a - 1, b - 1] += 1

plt.figure(figsize=(9, 8))
plt.imshow(co, cmap="viridis")
plt.colorbar(label="co-occurrences")
plt.title("White-ball pair co-occurrence — texture, but no structure")
plt.xlabel("number"); plt.ylabel("number")
plt.show()

## 3 · Is any of that real? The chi-square test

The frequency charts *look* interesting — some bars tower over others. The
**chi-square goodness-of-fit test** asks: are these deviations bigger than what pure
uniform randomness would produce anyway?

- **Null hypothesis:** every number is equally likely (the draw is uniform).
- **p-value > 0.05:** the observed "hot/cold" pattern is entirely consistent with pure chance.

This is the statistical heart of the notebook. Everything after this is us building an ML
pipeline *in full knowledge* of what this test says.

In [ ]:
chi_w, p_w = stats.chisquare(white_counts.values)
chi_pb, p_pb = stats.chisquare(pb_counts.values)

print(f"White balls : chi²={chi_w:8.2f}  p-value={p_w:.4f}")
print(f"Powerball   : chi²={chi_pb:8.2f}  p-value={p_pb:.4f}")
print()
for name, p in [("white balls", p_w), ("Powerball", p_pb)]:
    verdict = ("consistent with a UNIFORM RANDOM process — the 'hot numbers' are noise"
               if p > 0.05 else
               "deviates from uniform — before you get excited, suspect a data issue, "
               "the multiple-comparisons trap, or the era filter")
    print(f"→ {name}: {verdict}")

## 4 · Building the training dataset

Now the ML engineering. We frame this as a **sequence task**: given the last `WINDOW`
draws, predict the next one.

**Encoding.** Each draw becomes a **95-dim multi-hot vector**: positions 0–68 flag the five
white balls, positions 69–94 flag the Powerball. A training example is a `(WINDOW, 95)`
matrix; the target is the next draw's white multi-hot (for multi-label BCE loss) and the
Powerball index (for cross-entropy).

**Splitting.** Time-series data must be split **chronologically** — train on the past,
validate/test on the future. Shuffling before splitting would leak future information
into training, one of the most common real-world ML bugs.

In [ ]:
WINDOW = 32
DRAW_DIM = N_WHITE + N_PB  # 95

def encode(whites, pb):
    v = np.zeros(DRAW_DIM, dtype=np.float32)
    v[np.asarray(whites) - 1] = 1.0
    v[N_WHITE + pb - 1] = 1.0
    return v

encoded = np.stack([encode(row[:5], row[5])
                    for row in df[WHITE_COLS + ["pb"]].values])

X, y_white, y_pb = [], [], []
for i in range(WINDOW, len(encoded)):
    X.append(encoded[i - WINDOW:i])
    y_white.append(encoded[i, :N_WHITE])
    y_pb.append(int(df["pb"].iloc[i]) - 1)

X = torch.tensor(np.stack(X))
y_white = torch.tensor(np.stack(y_white))
y_pb = torch.tensor(y_pb)

# Chronological 80/10/10 split — no shuffling across the time boundary!
n = len(X)
n_train, n_val = int(n * 0.8), int(n * 0.1)
splits = {
    "train": slice(0, n_train),
    "val":   slice(n_train, n_train + n_val),
    "test":  slice(n_train + n_val, n),
}
for name, s in splits.items():
    print(f"{name:5s}: {len(X[s]):4d} examples")

from torch.utils.data import TensorDataset, DataLoader
loaders = {
    name: DataLoader(TensorDataset(X[s], y_white[s], y_pb[s]),
                     batch_size=64, shuffle=(name == "train"))
    for name, s in splits.items()
}

## 5 · The model: a small Transformer encoder

We use the same architecture family that powers modern LLMs, scaled way down:

- **Input projection** lifts each 95-dim draw vector into a 128-dim embedding
- **Learned positional embeddings** tell the model *where in the window* each draw sits
- **3 Transformer encoder layers** (4 attention heads each) let every draw attend to every other
- **Mean-pool + two heads**: 69 white-ball logits and 26 Powerball logits

Loss = `BCEWithLogitsLoss` on the white multi-hot (a multi-label problem: 5 of 69 are "on")
+ `CrossEntropyLoss` on the Powerball (a single-label problem).

> The same class lives in [`src/model.py`](https://github.com/jamelski1/pbml/blob/claude/powerball-prediction-ml-fxhi8i/src/model.py)
> so the future Gradio web app can reuse it. Keep the two in sync if you tweak it.

In [ ]:
class PowerballTransformer(nn.Module):
    def __init__(self, d_model=128, nhead=4, num_layers=3, dim_feedforward=256,
                 window=WINDOW, dropout=0.1):
        super().__init__()
        self.config = dict(d_model=d_model, nhead=nhead, num_layers=num_layers,
                           dim_feedforward=dim_feedforward, window=window,
                           dropout=dropout)
        self.input_proj = nn.Linear(DRAW_DIM, d_model)
        self.pos_emb = nn.Parameter(torch.zeros(1, window, d_model))
        nn.init.trunc_normal_(self.pos_emb, std=0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head_white = nn.Linear(d_model, N_WHITE)
        self.head_pb = nn.Linear(d_model, N_PB)

    def forward(self, x):
        h = self.input_proj(x) + self.pos_emb[:, : x.size(1)]
        h = self.encoder(h)
        h = self.norm(h.mean(dim=1))
        return self.head_white(h), self.head_pb(h)

model = PowerballTransformer().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"PowerballTransformer: {n_params:,} parameters on {device}")

## 6 · Training on the GPU

The real curriculum of this section is the **GPU training loop** you'd use on any project:

- `.to(device)` moves model and batches onto the GPU
- **Mixed precision** (`torch.autocast` + `GradScaler`) runs the matmuls in fp16/bf16 —
  on an A100/L4/T4 this is the standard trick for big speedups at scale
- **AdamW + gradient clipping**, tracking train *and* validation loss every epoch

Watch what the curves do: train loss falls while validation loss stalls (or rises).
That's the signature of a model **memorizing noise** — there is no signal to generalize from.
On a real dataset this gap is your overfitting alarm; here it's the whole story.

In [ ]:
EPOCHS, LR = 40, 3e-4
bce = nn.BCEWithLogitsLoss()
ce = nn.CrossEntropyLoss()
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scaler = torch.amp.GradScaler(enabled=device.type == "cuda")

def batch_loss(xb, yw, yp):
    white_logits, pb_logits = model(xb)
    return bce(white_logits, yw) + ce(pb_logits, yp)

history = {"train": [], "val": []}
for epoch in range(1, EPOCHS + 1):
    model.train(); tot = 0.0
    for xb, yw, yp in loaders["train"]:
        xb, yw, yp = xb.to(device), yw.to(device), yp.to(device)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type,
                            enabled=device.type == "cuda"):
            loss = batch_loss(xb, yw, yp)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        tot += loss.item() * len(xb)
    history["train"].append(tot / len(loaders["train"].dataset))

    model.eval(); tot = 0.0
    with torch.no_grad():
        for xb, yw, yp in loaders["val"]:
            xb, yw, yp = xb.to(device), yw.to(device), yp.to(device)
            tot += batch_loss(xb, yw, yp).item() * len(xb)
    history["val"].append(tot / len(loaders["val"].dataset))

    if epoch % 5 == 0 or epoch == 1:
        print(f"epoch {epoch:3d} | train {history['train'][-1]:.4f} "
              f"| val {history['val'][-1]:.4f}")

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(history["train"], label="train loss")
plt.plot(history["val"], label="validation loss")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.title("Falling train loss + flat/rising val loss = memorizing noise")
plt.tight_layout(); plt.show()

## 7 · The honest backtest ⚖️

**This is the most important section of the notebook.** A model that outputs lottery numbers
*looks* like it works — it always produces an answer. The only fair question is:

> On draws the model never saw, does it match more numbers than random guessing?

We walk forward through the held-out test draws and score three strategies:

| Strategy | How it picks |
|---|---|
| 🤖 **Transformer** | model's top-5 white logits + top-1 Powerball |
| 📊 **Frequency picker** | the 5 historically hottest whites + hottest Powerball |
| 🎲 **Random** | uniform random tickets (averaged over 500 trials) |

Math says a random ticket matches **5 × 5/69 ≈ 0.362** white balls per draw and hits the
Powerball **1/26 ≈ 3.85%** of the time. If the Transformer had found real signal, it would
beat those numbers. Spoiler: it won't — and *that's the finding*.

In [ ]:
model.eval()
test_X, test_yw, test_yp = X[splits["test"]], y_white[splits["test"]], y_pb[splits["test"]]

with torch.no_grad():
    wl, pl = model(test_X.to(device))
    model_white = wl.topk(5, dim=1).indices.cpu()   # top-5 white picks per draw
    model_pb = pl.argmax(dim=1).cpu()

actual_white_sets = [set(torch.nonzero(r).squeeze(1).tolist()) for r in test_yw]

def score(pred_white_rows, pred_pb):
    w_hits = np.mean([len(set(p.tolist()) & a)
                      for p, a in zip(pred_white_rows, actual_white_sets)])
    pb_hits = float((pred_pb == test_yp).float().mean())
    return w_hits, pb_hits

# 🤖 Transformer
tf_w, tf_pb = score(model_white, model_pb)

# 📊 Frequency picker (uses training-era counts only — no peeking at the test set)
train_end = splits["train"].stop + WINDOW
freq_w = pd.Series(df[WHITE_COLS].values[:train_end].ravel()).value_counts()
freq_white = torch.tensor(sorted(freq_w.nlargest(5).index)) - 1
freq_pb_pick = int(df["pb"].iloc[:train_end].value_counts().idxmax()) - 1
fr_w, fr_pb = score(freq_white.repeat(len(test_X), 1),
                    torch.full((len(test_X),), freq_pb_pick))

# 🎲 Random baseline, 500 simulated ticket-per-draw trials
rng = np.random.default_rng(SEED)
rand_w_trials, rand_pb_trials = [], []
for _ in range(500):
    picks = [rng.choice(N_WHITE, 5, replace=False) for _ in range(len(test_X))]
    rand_w_trials.append(np.mean([len(set(p) & a)
                                  for p, a in zip(picks, actual_white_sets)]))
    rand_pb_trials.append(np.mean(rng.integers(0, N_PB, len(test_X)) ==
                                  test_yp.numpy()))
rd_w, rd_pb = np.mean(rand_w_trials), np.mean(rand_pb_trials)

results = pd.DataFrame({
    "white matches / draw": [tf_w, fr_w, rd_w, 5 * 5 / N_WHITE],
    "Powerball hit rate":   [tf_pb, fr_pb, rd_pb, 1 / N_PB],
}, index=["🤖 Transformer", "📊 Frequency picker", "🎲 Random (500 trials)",
          "📐 Theoretical chance"]).round(4)
results

### Reading the scoreboard

All the strategies cluster around **0.36 white matches per draw** and **~4% Powerball
hits** — the theoretical chance line. The Transformer's tiny wobble above or below random is
itself just luck; rerun with a different seed and it lands on the other side.

**This is a genuinely successful experiment.** We asked "is there learnable structure in
Powerball history?", built a fair test, and got a clear answer: **no**. Being able to produce
— and trust — a negative result is a core ML skill. It's also why nobody should ever pay for
lottery predictions. 🎓

## 8 · Generate your numbers 🎟️

The model can't beat random — but random is exactly what a lottery ticket should be!
We sample 5 distinct white balls and a Powerball from the model's (noise-shaped) output
distribution. Any ticket has the same 1-in-292,201,338 jackpot odds as any other.

In [ ]:
@torch.no_grad()
def sample_play(temperature=1.0):
    history = torch.tensor(encoded[-WINDOW:]).unsqueeze(0).to(device)
    wl, pl = model(history)
    white_p = torch.softmax(wl[0] / temperature, dim=0)
    pb_p = torch.softmax(pl[0] / temperature, dim=0)
    whites = torch.multinomial(white_p, 5, replacement=False)
    pb = torch.multinomial(pb_p, 1)
    return sorted((whites + 1).tolist()), int(pb.item()) + 1

print("Your (statistically meaningless, mathematically fair) tickets:\n")
for i in range(3):
    whites, pb = sample_play()
    print(f"  🎟️  Ticket {i+1}: {' '.join(f'{w:2d}' for w in whites)}  ⚡ Powerball: {pb}")

## 9 · Publish to the Hugging Face Hub 🤗

Publishing a model — weights, config, and an honest model card — is a skill worth
practicing on any project. Flip `PUSH_TO_HUB = True`, run the cell, and paste a
**write** token from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

In [ ]:
PUSH_TO_HUB = False          # ← flip to True when you're ready
HF_REPO_ID = "YOUR_HF_USERNAME/powerball-transformer"   # ← edit me

import json as _json, os
os.makedirs("export", exist_ok=True)
torch.save(model.state_dict(), "export/pytorch_model.bin")
with open("export/config.json", "w") as f:
    _json.dump(model.config, f, indent=2)

MODEL_CARD = f'''---
license: mit
tags: [educational, pytorch, transformer, time-series]
---

# powerball-transformer 🎱

A tiny Transformer encoder trained on historical Powerball draws — built as an
**educational end-to-end ML project** ([source repo](https://github.com/jamelski1/pbml)).

## ⚠️ This model cannot predict the lottery

Powerball draws are independent random events. A walk-forward backtest in the training
notebook shows this model matches **{tf_w:.3f}** white balls per draw vs. **{5*5/N_WHITE:.3f}**
expected by pure chance — i.e. it performs exactly like random guessing, as the laws of
probability require. It is published to demonstrate the training → evaluation → deployment
workflow, and to power a demo number-generator app.

## Architecture
{n_params:,} params — Linear(95→128) → 3× TransformerEncoderLayer(d=128, heads=4) →
mean-pool → dual heads (69 white logits, 26 Powerball logits). Input: the last {WINDOW}
draws as 95-dim multi-hot vectors.
'''
with open("export/README.md", "w") as f:
    f.write(MODEL_CARD)
print("Export folder ready:", os.listdir("export"))

if PUSH_TO_HUB:
    from huggingface_hub import notebook_login, HfApi
    notebook_login()
    api = HfApi()
    api.create_repo(HF_REPO_ID, exist_ok=True, repo_type="model")
    api.upload_folder(folder_path="export", repo_id=HF_REPO_ID)
    print(f"🚀 pushed → https://huggingface.co/{HF_REPO_ID}")

## 10 · What you just learned — and what's next

✅ Pulled and cleaned real-world open data (and handled a rules-change regime shift)
✅ Ran a skeptical EDA and a chi-square test instead of trusting pretty charts
✅ Encoded a sequence problem, split it chronologically, and avoided data leakage
✅ Trained a Transformer with mixed precision on a GPU
✅ Backtested honestly against a random baseline — and trusted a negative result
✅ Packaged and published a model with an honest model card

**Next steps for the pbml project:**
- 🌐 Build the web interface: a Gradio app on Hugging Face Spaces that loads this model
  and generates tickets (a starter lives in [`app/app.py`](https://github.com/jamelski1/pbml/blob/claude/powerball-prediction-ml-fxhi8i/app/app.py))
- 🧪 Experiment: bigger windows, different architectures, other loss functions — verify
  the backtest verdict never changes (it won't, and now you can prove why)
- 🔁 Reuse this exact pipeline skeleton on data that *does* have signal — sales,
  weather, text — where these same skills pay off for real